In [1]:
import re
from collections import defaultdict
from functools import cache
from typing import Annotated, Literal

import pandas as pd
import pulp
from pydantic import BaseModel, BeforeValidator, TypeAdapter

In [2]:
bracket_raw = pd.read_csv("00_bracket.csv")
lookup_raw = pd.read_csv("00_lookup.csv")
ratings_raw = pd.read_csv("01_ratings.csv")

In [3]:
ratings = (
    lookup_raw.merge(ratings_raw, how="left", left_on="eloratings_name", right_on="team")
    .set_index("fifa_code")["rating"]
    .to_dict()
)

In [4]:
def parse_team(s):
    if re.match(R"[WL]\d+", s):
        return PrevMatch.from_string(s)
    return s


class PrevMatch(BaseModel):
    match_number: int
    team: Literal["W", "L"]

    @classmethod
    def from_string(cls, s: str):
        return PrevMatch(match_number=int(s[1:]), team=s[0])


class BracketMatch(BaseModel):
    match_number: int
    home: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    away: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    stage: str

In [5]:
matches = {
    match.match_number: match
    for match in TypeAdapter(list[BracketMatch]).validate_python(bracket_raw.to_dict(orient="records"))
}

In [6]:
from typing import Self


class MatchTeam(BaseModel):
    team: str
    source: PrevMatch | None = None

    def replace_source(self, source: PrevMatch) -> Self:
        return self.model_copy(update={"source": source})


@cache
def possible_teams_for_match(match_number: int) -> list[MatchTeam]:
    match = matches[match_number]
    out = []
    if not isinstance(match.home, str):
        out.extend(mt.replace_source(match.home) for mt in possible_teams_for_match(match.home.match_number))
    else:
        out.append(MatchTeam(team=match.home))

    if not isinstance(match.away, str):
        out.extend(mt.replace_source(match.away) for mt in possible_teams_for_match(match.away.match_number))
    else:
        out.append(MatchTeam(team=match.away))

    return out

In [7]:
prob = pulp.LpProblem("Bracket", pulp.LpMinimize)

winner_vars = defaultdict(dict)
participant_vars = defaultdict(dict)

ratings_difference = 0

for match in sorted(matches):
    match_winners = []
    match_participants = []
    for match_team in possible_teams_for_match(match):
        team = match_team.team
        source = match_team.source
        rating = ratings[team]
        winner_var = prob.add_variable(f"m{match:03d}_{team}_winner", 0, 1, pulp.LpInteger)
        participant_var = prob.add_variable(f"m{match:03d}_{team}_participant", 0, 1, pulp.LpInteger)

        ratings_difference += (participant_var - 2 * winner_var) * rating

        prob += participant_var >= winner_var
        if source:
            source_part_var = participant_vars[source.match_number][team]
            prob += source_part_var >= participant_var
            source_winner_var = winner_vars[source.match_number][team]
            if source.team == "W":
                prob += source_winner_var == participant_var
            else:
                prob += (1 - source_winner_var) >= participant_var
        else:
            prob += participant_var == 1

        winner_vars[match][team] = winner_var
        participant_vars[match][team] = participant_var

        match_winners.append(winner_var)
        match_participants.append(participant_var)
    prob += pulp.lpSum(match_winners) == 1
    prob += pulp.lpSum(match_participants) == 2

prob += ratings_difference

winner_vars = dict(winner_vars)
participant_vars = dict(participant_vars)

In [8]:
prob.solve()

winners = {}
participants = defaultdict(list)

for var in prob.variables():
    if var.name == "__dummy":
        continue
    if var.value() != 1:
        continue
    match_raw, team, kind = var.name.split("_")
    match = int(match_raw[1:])
    if kind == "winner":
        winners[match] = team
    elif kind == "participant":
        participants[match].append(team)
    else:
        raise ValueError(var.name)

participants = dict(participants)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/adam/repos/wc26-tipping/.venv/lib64/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/648c07fbe2d84d47b950e3b96e57ec7a-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/648c07fbe2d84d47b950e3b96e57ec7a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 613 COLUMNS
At line 3206 RHS
At line 3815 BOUNDS
At line 4200 ENDATA
Problem MODEL has 608 rows, 384 columns and 1440 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -5303 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 96 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 29 strengthened rows, 0 substitutions
Cgl0003I 0 fi

In [9]:
for match in sorted(matches):
    winner = winners[match]
    home, away = participants[match]
    print(
        f"{match:3} {matches[match].stage:7} {'👑' if winner == home else '  '}{home} ({ratings[home]}) vs {'👑' if winner == away else '  '}{away} ({ratings[away]})"
    )

 73 R32     👑CAN (1764) vs   RSA (1559)
 74 R32     👑GER (1916) vs   PAR (1815)
 75 R32       MAR (1877) vs 👑NED (1980)
 76 R32     👑BRA (2009) vs   JPN (1910)
 77 R32     👑FRA (2123) vs   SWE (1742)
 78 R32       CIV (1743) vs 👑NOR (1918)
 79 R32       ECU (1902) vs 👑MEX (1912)
 80 R32       COD (1712) vs 👑ENG (2038)
 81 R32       BIH (1622) vs 👑USA (1781)
 82 R32     👑BEL (1884) vs   SEN (1842)
 83 R32       CRO (1905) vs 👑POR (1990)
 84 R32       AUT (1800) vs 👑ESP (2144)
 85 R32       ALG (1785) vs 👑SUI (1914)
 86 R32     👑ARG (2148) vs   CPV (1622)
 87 R32     👑COL (2004) vs   GHA (1575)
 88 R32     👑AUS (1800) vs   EGY (1742)
 89 R16     👑FRA (2123) vs   GER (1916)
 90 R16       CAN (1764) vs 👑NED (1980)
 91 R16     👑BRA (2009) vs   NOR (1918)
 92 R16     👑ENG (2038) vs   MEX (1912)
 93 R16     👑ESP (2144) vs   POR (1990)
 94 R16     👑BEL (1884) vs   USA (1781)
 95 R16     👑ARG (2148) vs   AUS (1800)
 96 R16     👑COL (2004) vs   SUI (1914)
 97 quarter 👑FRA (2123) vs   NED (1980)
